In [1]:
import os
import shutil

src_dir = None
req_file = None

for dirname, dirs, files in os.walk('/kaggle/input'):
    if 'src' in dirs and 'chakramodel' in dirname.lower():
        src_dir = os.path.join(dirname, 'src')
    if 'requirements.txt' in files:
        req_file = os.path.join(dirname, 'requirements.txt')

if src_dir and req_file:
    print(f"Found code at: {src_dir}")
    if os.path.exists('./src'):
        shutil.rmtree('./src')
    shutil.copytree(src_dir, './src')
    print("Copied 'src' to working directory!")
    print("Installing requirements...")
    os.system(f"pip install -r {req_file} -q")
    print("Setup complete!")
else:
    print("Error: Could not find uploaded files. Make sure both datasets are attached.")

Found code at: /kaggle/input/datasets/gokulrocky/chakramodel-kaggle-code/src
Copied 'src' to working directory!
Installing requirements...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.1/46.1 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 24.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 65.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.2/66.2 kB 4.3 MB/s eta 0:00:00
Setup complete!


In [2]:
%%writefile src/chakra_transformer/train_transformer.py
import argparse
import sys
from pathlib import Path
import numpy as np
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

sys.path.insert(0, str(Path(__file__).resolve().parent.parent))
from train_pranet import KvasirSEGDataset, DiceFocalLoss
from metrics.seg_metrics import dice, iou
from transformer_segmenter import ChakraTransformerSegmenter
from topo_loss import TopologicalLoss

def main():
    parser = argparse.ArgumentParser(description="Train ChakraTransformer")
    parser.add_argument('--batch-size', type=int, default=16)
    parser.add_argument('--epochs', type=int, default=100)
    parser.add_argument('--lr', type=float, default=1e-4)
    parser.add_argument('--num-workers', type=int, default=2)
    args = parser.parse_args()

    print("=== ChakraTransformer Training (High-Accuracy Research Track) ===")
    print(f"Batch Size: {args.batch_size}, Epochs: {args.epochs}, LR: {args.lr}")
    print(f"GPUs available: {torch.cuda.device_count()}")

    assert torch.cuda.is_available(), "CUDA must be available!"
    device = torch.device("cuda")
    model = ChakraTransformerSegmenter().to(device)

    # ---- MULTI-GPU: automatically uses both T4s if available ----
    if torch.cuda.device_count() > 1:
        print(f"Using {torch.cuda.device_count()} GPUs with DataParallel!")
        model = nn.DataParallel(model)
    # -------------------------------------------------------------

    optimizer = optim.AdamW(model.parameters(), lr=args.lr, weight_decay=1e-4)
    criterion_dice = DiceFocalLoss()
    criterion_topo = TopologicalLoss(lam=0.1)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=args.epochs, eta_min=args.lr * 0.01)

    print(f"Model initialized on {device}. Loading dataset...")

    root = Path(__file__).resolve().parent.parent.parent
    images_dir = root / "data" / "kvasir-seg" / "images"
    masks_dir  = root / "data" / "kvasir-seg" / "masks"

    if not images_dir.exists():
        print("[ERROR] Kvasir-SEG not found!")
        sys.exit(1)

    img_size = 384
    full_dataset = KvasirSEGDataset(images_dir, masks_dir, img_size=img_size, augment=True,  use_gt_bbox_crop=True)
    val_dataset  = KvasirSEGDataset(images_dir, masks_dir, img_size=img_size, augment=False, use_gt_bbox_crop=True)

    all_paths = sorted([p for p in images_dir.glob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".tif"}])
    n_train = int(0.7 * len(all_paths))
    n_val   = int(0.1 * len(all_paths))

    train_set = torch.utils.data.Subset(full_dataset, range(n_train))
    val_set   = torch.utils.data.Subset(val_dataset,  range(n_train, n_train + n_val))

    train_loader = DataLoader(train_set, batch_size=args.batch_size, shuffle=True,  num_workers=args.num_workers, pin_memory=True)
    val_loader   = DataLoader(val_set,   batch_size=args.batch_size, shuffle=False, num_workers=args.num_workers, pin_memory=True)

    weights_dir = root / "weights"
    weights_dir.mkdir(exist_ok=True)
    best_dice = 0.0
    best_path = weights_dir / "chakra_transformer_best.pth"

    for epoch in range(1, args.epochs + 1):
        model.train()
        train_loss = 0.0

        print(f"\n--- Epoch {epoch}/{args.epochs} ---")
        pbar = tqdm(train_loader, desc="Training")
        for imgs, masks in pbar:
            imgs, masks = imgs.to(device), masks.to(device)
            optimizer.zero_grad()
            logits = model(imgs)
            loss = criterion_dice(logits, masks)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
            pbar.set_postfix({'loss': f"{loss.item():.4f}"})

        scheduler.step()
        train_loss /= len(train_loader)

        model.eval()
        val_dice_scores, val_iou_scores = [], []
        with torch.no_grad():
            for imgs, masks in val_loader:
                imgs   = imgs.to(device)
                logits = model(imgs)
                probs  = torch.sigmoid(logits).cpu().numpy()
                masks_np = masks.numpy()
                for i in range(len(probs)):
                    pred = (probs[i, 0] > 0.5).astype(np.uint8) * 255
                    gt   = (masks_np[i, 0] > 0.5).astype(np.uint8) * 255
                    val_dice_scores.append(dice(pred, gt))
                    val_iou_scores.append(iou(pred, gt))

        mean_dice = float(np.mean(val_dice_scores))
        mean_iou  = float(np.mean(val_iou_scores))
        lr_now    = optimizer.param_groups[0]["lr"]
        print(f"  --> Epoch [{epoch:3d}/{args.epochs}] | Train Loss: {train_loss:.4f} | Val Dice: {mean_dice:.4f} | Val IoU: {mean_iou:.4f} | LR: {lr_now:.6f}", flush=True)

        if mean_dice > best_dice:
            best_dice = mean_dice
            torch.save(model.state_dict(), best_path)
            print(f"  *** New best Dice: {best_dice:.4f} -> saved to {best_path}")

    print(f"\nTraining Complete! Best Val Dice: {best_dice:.4f}. Weights: {best_path}")

if __name__ == '__main__':
    main()

Overwriting src/chakra_transformer/train_transformer.py


In [3]:
!pip install gudhi

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 32.1 MB/s eta 0:00:00


In [4]:
import os

os.system("rm -rf ./data")
os.makedirs("./data", exist_ok=True)

kvasir_dir = None
for dirname, dirs, files in os.walk('/kaggle/input'):
    if 'images' in dirs and 'masks' in dirs and 'kvasir' in dirname.lower():
        kvasir_dir = dirname
        break

if kvasir_dir:
    print(f"Found Kvasir-SEG dataset at: {kvasir_dir}")
    os.system(f"ln -s {kvasir_dir} ./data/kvasir-seg")
    print("Starting training...")
    os.system("python src/chakra_transformer/train_transformer.py --epochs 100 --batch-size 16")
else:
    print("ERROR: Could not find the Kvasir-SEG dataset. Make sure datasets are attached!")

Found Kvasir-SEG dataset at: /kaggle/input/datasets/gokulrocky/chakramodel-evaluation-datasets/kvasir-seg
Starting training...


=== ChakraTransformer Training (High-Accuracy Research Track) ===
Batch Size: 16, Epochs: 100, LR: 0.0001
GPUs available: 2
Using 2 GPUs with DataParallel!
Model initialized on cuda. Loading dataset...

--- Epoch 1/100 ---


Training: 100%|██████████| 44/44 [02:26<00:00,  3.32s/it, loss=0.3345]


  --> Epoch [  1/100] | Train Loss: 0.5134 | Val Dice: 0.9154 | Val IoU: 0.8503 | LR: 0.000100
  *** New best Dice: 0.9154 -> saved to /kaggle/working/weights/chakra_transformer_best.pth

--- Epoch 2/100 ---


Training: 100%|██████████| 44/44 [02:29<00:00,  3.40s/it, loss=0.2809]


...

  --> Epoch [ 98/100] | Train Loss: 0.0289 | Val Dice: 0.9411 | Val IoU: 0.8948 | LR: 0.000001

--- Epoch 99/100 ---


Training: 100%|██████████| 44/44 [02:29<00:00,  3.41s/it, loss=0.0347]


  --> Epoch [ 99/100] | Train Loss: 0.0288 | Val Dice: 0.9411 | Val IoU: 0.8946 | LR: 0.000001

--- Epoch 100/100 ---


Training: 100%|██████████| 44/44 [02:29<00:00,  3.41s/it, loss=0.0283]


  --> Epoch [100/100] | Train Loss: 0.0288 | Val Dice: 0.9413 | Val IoU: 0.8950 | LR: 0.000001

Training Complete! Best Val Dice: 0.9445. Weights: /kaggle/working/weights/chakra_transformer_best.pth
